# YouTube Speaker Diarization

This notebook downloads a YouTube video, transcribes it with speaker labels using pyannote.audio, and outputs a formatted transcript.

**Setup:**
1. Runtime → Change runtime type → T4 GPU
2. Get a HuggingFace token from https://huggingface.co/settings/tokens
3. Accept pyannote model terms: https://huggingface.co/pyannote/speaker-diarization-3.1

In [ ]:
#@title 1. Install Dependencies
!pip install -q pyannote.audio yt-dlp youtube-transcript-api

In [ ]:
#@title 2. Configuration
import re

YOUTUBE_URL = "https://www.youtube.com/watch?v=a0UCcZWx9ao"  #@param {type:"string"}
HF_TOKEN = ""  #@param {type:"string"}
START_MIN = 3  #@param {type:"integer"}
END_MIN = 15  #@param {type:"integer"}

START_SEC = START_MIN * 60
END_SEC = END_MIN * 60

def extract_video_id(url):
    match = re.search(r'(?:v=|youtu\.be/)([a-zA-Z0-9_-]{11})', url)
    return match.group(1) if match else url

VIDEO_ID = extract_video_id(YOUTUBE_URL)
print(f"Video ID: {VIDEO_ID}")
print(f"Time range: {START_MIN}:00 - {END_MIN}:00")

In [ ]:
#@title 3. Download Audio from YouTube
import yt_dlp
import os
import glob

# Clean up any previous downloads
for f in glob.glob('audio*'):
    os.remove(f)

ydl_opts = {
    'format': 'bestaudio/best',
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'wav',
    }],
    'outtmpl': 'audio.%(ext)s',
    'quiet': False,
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(YOUTUBE_URL, download=True)
    video_title = info.get('title', 'Unknown')

# Find the wav file
audio_file = glob.glob('audio*.wav')[0]
print(f"\nDownloaded: {video_title}")
print(f"Audio file: {audio_file}")

In [ ]:
#@title 4. Trim Audio to Time Range
import subprocess

trim_cmd = [
    'ffmpeg', '-y',
    '-i', audio_file,
    '-ss', str(START_SEC),
    '-to', str(END_SEC),
    '-c', 'copy',
    'audio_trimmed.wav',
    '-loglevel', 'error'
]

subprocess.run(trim_cmd, check=True)
print(f"Trimmed to {START_MIN}:00 - {END_MIN}:00")
print(f"Duration: {END_MIN - START_MIN} minutes")

In [ ]:
#@title 5. Run Speaker Diarization (pyannote)
from pyannote.audio import Pipeline
import torch

if not HF_TOKEN:
    raise ValueError("Please set HF_TOKEN in the Configuration cell!")

print("Loading pyannote pipeline (this may take a minute)...")
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=HF_TOKEN
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
pipeline.to(device)

print("Running diarization...")
diarization = pipeline("audio_trimmed.wav")

speakers_found = set(label for _, _, label in diarization.itertracks(yield_label=True))
print(f"\nDone! Speakers found: {len(speakers_found)}")

In [ ]:
#@title 6. Fetch YouTube Transcript
from youtube_transcript_api import YouTubeTranscriptApi

api = YouTubeTranscriptApi()
transcript_data = api.fetch(VIDEO_ID)

# Filter to time range and adjust timestamps to be relative to trimmed audio
segments = []
for s in transcript_data:
    seg_start = s.start
    seg_end = s.start + s.duration
    
    # Include if segment overlaps with our time range
    if seg_end >= START_SEC and seg_start <= END_SEC:
        segments.append({
            'text': s.text,
            'start': s.start - START_SEC,  # Relative to trimmed audio
            'end': seg_end - START_SEC,
            'start_absolute': s.start,  # Keep original for display
        })

print(f"Transcript segments in range: {len(segments)}")

In [ ]:
#@title 7. Merge Diarization with Transcript
def get_speaker_at_time(diarization, time_sec):
    """Find which speaker is talking at a given time."""
    for turn, _, speaker in diarization.itertracks(yield_label=True):
        if turn.start <= time_sec <= turn.end:
            return speaker
    return None

# Assign speakers to transcript segments
for seg in segments:
    # Use midpoint of segment to determine speaker
    mid_time = (seg['start'] + seg['end']) / 2
    # Clamp to valid range
    mid_time = max(0, mid_time)
    seg['speaker'] = get_speaker_at_time(diarization, mid_time)

# Get unique speakers in order of appearance
speakers_seen = []
for seg in segments:
    if seg['speaker'] and seg['speaker'] not in speakers_seen:
        speakers_seen.append(seg['speaker'])

# Create friendly names
speaker_names = {spk: f"Speaker {i+1}" for i, spk in enumerate(speakers_seen)}

print(f"Unique speakers identified: {len(speaker_names)}")
for orig, name in speaker_names.items():
    print(f"  {orig} → {name}")

In [ ]:
#@title 8. Generate Formatted Transcript
def format_time(seconds):
    """Format seconds as MM:SS"""
    total_sec = int(seconds)
    return f"{total_sec // 60:02d}:{total_sec % 60:02d}"

output_lines = [
    f"# {video_title}",
    "",
    f"**Video:** {YOUTUBE_URL}",
    f"**Time Range:** {START_MIN}:00 - {END_MIN}:00",
    f"**Speakers Identified:** {len(speaker_names)}",
    "",
    "## Transcript",
    ""
]

current_speaker = None
for seg in segments:
    speaker = speaker_names.get(seg['speaker'], 'Unknown')
    timestamp = format_time(seg['start_absolute'])  # Use absolute time for display
    
    if speaker != current_speaker:
        current_speaker = speaker
        output_lines.append(f"\n**{speaker}** [{timestamp}]")
    
    # Clean up >> markers from YouTube transcript
    text = re.sub(r'^>>+\s*', '', seg['text'])
    output_lines.append(text)

transcript_output = "\n".join(output_lines)

# Save to file
output_filename = f"transcript_{VIDEO_ID}_{START_MIN}-{END_MIN}min.md"
with open(output_filename, 'w') as f:
    f.write(transcript_output)

print(f"Saved to {output_filename}")
print("\n" + "="*50)
print(transcript_output[:3000])
if len(transcript_output) > 3000:
    print("\n[... truncated ...]")

In [ ]:
#@title 9. Download Result
from google.colab import files
files.download(output_filename)

In [ ]:
#@title 10. (Optional) View Diarization Timeline
print("Speaker Timeline:")
print("="*60)
for turn, _, speaker in diarization.itertracks(yield_label=True):
    start_abs = turn.start + START_SEC
    end_abs = turn.end + START_SEC
    name = speaker_names.get(speaker, speaker)
    print(f"{format_time(start_abs)} - {format_time(end_abs)}: {name}")